# SupportGen · 05 — Supporting classifiers (queue, priority, type) + inferred sentiment
These components are **predictive, not generative**. They give the support agent structured
information next to the generated reply. We use simple, transparent baselines:

**TF-IDF + linear classifier** (Logistic Regression or Linear SVM)
* TF-IDF turns text into a sparse vector: each word / character n-gram gets a weight that is high
  when it is frequent in this ticket but rare across all tickets.
* A linear model learns one weight per feature per class — fast, strong on text, easy to explain.

They are trained on the **same train split** as the generator, tuned on validation, and reported
**once** on the held-out test split. Because the labels are imbalanced, we report **macro-F1**
(every class counts equally) next to accuracy, and compare against trivial baselines.

**Runtime:** CPU is enough. ~10–20 minutes.

## 0. Setup — Google Drive & project folder
Every SupportGen notebook starts with this same cell. It mounts Google Drive so that all
notebooks share one project folder (`MyDrive/SupportGen`): notebook 01 writes the cleaned
splits there, notebooks 02–06 read them. Nothing ever needs to be uploaded from your laptop.

Set `USE_DRIVE = False` only for a throw-away run — files then live in `/content` and are lost
when the Colab runtime disconnects.

In [ ]:
# ── Setup (identical in every SupportGen notebook) ──────────────────────────
USE_DRIVE = True   # False → keep everything in /content (deleted when the runtime ends)

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/SupportGen"
else:
    BASE = "/content/SupportGen"
os.makedirs(BASE, exist_ok=True)
os.environ["SUPPORTGEN_BASE"] = BASE   # read by supportgen_common.py
sys.path.insert(0, BASE)
print("Project folder:", BASE)

In [ ]:
# Colab already ships torch, transformers, accelerate, scikit-learn, pandas, matplotlib.
# These are the extras SupportGen needs (already-installed packages are skipped).
!pip -q install sentencepiece

In [ ]:
# Shared helpers were written to Drive by 01_dataset_preparation.ipynb
if not os.path.exists(os.path.join(BASE, "supportgen_common.py")):
    raise FileNotFoundError(
        f"supportgen_common.py not found in {BASE}. Run 01_dataset_preparation.ipynb first "
        "(with the same USE_DRIVE setting).")
import importlib, supportgen_common as sg
importlib.reload(sg)
sg.set_seed()
print("Device:", sg.get_device())

In [ ]:
import json, joblib, sklearn
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

train_df, val_df, test_df = sg.load_split("train"), sg.load_split("val"), sg.load_split("test")
text = lambda d: (d.subject.fillna("") + " . " + d.body).tolist()
TARGETS = ["queue_group", "priority", "type"]
for t in TARGETS:
    print(f"{t}: train label distribution")
    display(train_df[t].value_counts(dropna=False).to_frame("rows").T)

## 1. Model selection on the validation split
For each target we try both classifiers with several regularisation strengths `C`
and `class_weight="balanced"` (rare classes get a larger weight in the loss, countering imbalance).
The combination with the best **validation macro-F1** is selected.

In [ ]:
def features():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=100_000, sublinear_tf=True)),
    ])

CANDIDATES = {
    "logreg": lambda C: LogisticRegression(C=C, max_iter=3000, class_weight="balanced"),
    "linear_svm": lambda C: LinearSVC(C=C, class_weight="balanced"),
}
C_GRID = [0.1, 0.5, 1.0, 4.0]

selection = {}
for target in TARGETS:
    tr, va = train_df.dropna(subset=[target]), val_df.dropna(subset=[target])
    feats = features().fit(text(tr))                 # fit vocabulary once per target (train only)
    Xtr, Xva = feats.transform(text(tr)), feats.transform(text(va))
    rows = []
    for name, make in CANDIDATES.items():
        for C in C_GRID:
            clf = make(C).fit(Xtr, tr[target])
            rows.append({"model": name, "C": C, "val_macro_f1": f1_score(va[target], clf.predict(Xva), average="macro")})
    res = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
    selection[target] = res.iloc[0].to_dict()
    print(f"\n{target}: best = {selection[target]}")
    display(res.head(4))

## 2. Final models → held-out test set
The selected configuration is refit on the training split. A Linear SVM has no probabilities, so it is
wrapped in `CalibratedClassifierCV` — the web app can then show a confidence score. Reported test
metrics are for **exactly the pipeline that is saved and deployed**.

Trivial baselines for context:
* **majority** — always predicts the most common class (shows why accuracy alone is misleading),
* **stratified random** — guesses following the training label distribution.

In [ ]:
cls_metrics, LABELS = {}, {}
os.makedirs(sg.CLASSIFIER_DIR, exist_ok=True)
for target in TARGETS:
    tr, te = train_df.dropna(subset=[target]), test_df.dropna(subset=[target])
    best = selection[target]
    base_clf = CANDIDATES[best["model"]](best["C"])
    clf = CalibratedClassifierCV(base_clf, cv=3) if best["model"] == "linear_svm" else base_clf
    pipe = Pipeline([("features", features()), ("clf", clf)]).fit(text(tr), tr[target])
    pred = pipe.predict(text(te))

    p, r, f, _ = precision_recall_fscore_support(te[target], pred, average="macro", zero_division=0)
    m = {"model": best["model"], "C": best["C"], "val_macro_f1": best["val_macro_f1"],
         "test_n": len(te), "accuracy": accuracy_score(te[target], pred),
         "macro_precision": p, "macro_recall": r, "macro_f1": f,
         "weighted_f1": f1_score(te[target], pred, average="weighted"),
         "per_class": classification_report(te[target], pred, output_dict=True, zero_division=0)}
    for name, strat in [("majority", "most_frequent"), ("stratified_random", "stratified")]:
        d = DummyClassifier(strategy=strat, random_state=sg.SEED).fit(text(tr), tr[target]).predict(text(te))
        m[f"baseline_{name}"] = {"accuracy": accuracy_score(te[target], d), "macro_f1": f1_score(te[target], d, average="macro")}

    labels = list(pipe.classes_)
    cm = confusion_matrix(te[target], pred, labels=labels)
    m["labels"], m["confusion_matrix"] = labels, cm.tolist()
    cls_metrics[target] = m
    LABELS[target] = labels

    print("=" * 90, f"\n{target.upper()}  ({best['model']}, C={best['C']})")
    print(classification_report(te[target], pred, zero_division=0))
    print(f"majority baseline: acc={m['baseline_majority']['accuracy']:.3f}, macro-F1={m['baseline_majority']['macro_f1']:.3f}")

    fig, ax = plt.subplots(figsize=(min(4 + len(labels) * 0.6, 14),) * 2)
    ConfusionMatrixDisplay(confusion_matrix(te[target], pred, labels=labels, normalize="true"), display_labels=labels).plot(
        ax=ax, xticks_rotation=60, values_format=".2f", colorbar=False)
    ax.set_title(f"{target} — test confusion matrix (row-normalised)")
    plt.savefig(os.path.join(sg.PLOTS_DIR, f"cm_{target}.png"), dpi=130, bbox_inches="tight"); plt.show()

    joblib.dump(pipe, os.path.join(sg.CLASSIFIER_DIR, f"{target}.joblib"))

## 3. Summary table

In [ ]:
summary = pd.DataFrame({t: {k: cls_metrics[t][k] for k in ["model", "C", "test_n", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]}
                        | {"majority_acc": cls_metrics[t]["baseline_majority"]["accuracy"],
                           "majority_macro_f1": cls_metrics[t]["baseline_majority"]["macro_f1"]}
                        for t in TARGETS}).T
display(summary)

## 4. Sentiment — *inferred*, not evaluated
The dataset has **no sentiment labels**, so we cannot train or measure a sentiment classifier on it and
we do **not** report accuracy. Instead we apply a pretrained 3-class model
(`cardiffnlp/twitter-roberta-base-sentiment-latest`: negative / neutral / positive) as an *auxiliary
signal*. Below we only describe what it predicts on a test sample; this is exploratory, not ground truth.

In [ ]:
from transformers import pipeline
SENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sent = pipeline("sentiment-analysis", model=SENT_MODEL, device=0 if sg.get_device() == "cuda" else -1)
sample = test_df.sample(min(500, len(test_df)), random_state=sg.SEED)
preds = sent(text(sample), truncation=True, max_length=512, batch_size=32)
sample["sentiment"] = [p["label"].lower() for p in preds]
dist = sample.sentiment.value_counts(normalize=True).round(3)
display(dist.to_frame("share of sampled test tickets"))
print("Inferred sentiment by priority (row %, exploratory only):")
display((pd.crosstab(sample.priority, sample.sentiment, normalize="index") * 100).round(1))
cls_metrics["sentiment_inferred"] = {"model": SENT_MODEL, "n_sampled": len(sample), "distribution": dist.to_dict(),
                                     "note": "Pretrained model applied without ground-truth labels; not evaluated."}

## 5. Save classifiers and metrics
`models/classifiers/` holds one scikit-learn pipeline per target (`.joblib`) and `meta.json`.

⚠️ joblib files should be loaded with the **same scikit-learn version** they were saved with — it is
recorded in `meta.json`; install that version in the Mac backend environment.

In [ ]:
meta = {"targets": TARGETS, "labels": LABELS, "text_input": "subject + ' . ' + body",
        "sklearn_version": sklearn.__version__, "selection": selection,
        "sentiment_model": SENT_MODEL}
sg.save_json(meta, os.path.join(sg.CLASSIFIER_DIR, "meta.json"))
sg.save_json(cls_metrics, os.path.join(sg.RESULTS_DIR, "cls_metrics.json"))

# sanity check: reload and predict one ticket
demo = {"subject": "Refund not received", "body": "I requested a refund two weeks ago and still have not received it."}
for t in TARGETS:
    pipe = joblib.load(os.path.join(sg.CLASSIFIER_DIR, f"{t}.joblib"))
    proba = pipe.predict_proba([demo["subject"] + " . " + demo["body"]])[0]
    print(f"{t:12s} → {pipe.classes_[proba.argmax()]}  ({proba.max():.2f})")
print("\nsaved:", sorted(os.listdir(sg.CLASSIFIER_DIR)))

**Interpretation tips for the report:** compare macro-F1 against the majority baseline. On a synthetic
dataset, priority may be only weakly predictable from text — if so, say so; it is a valid finding.

➡️ **Next:** `06_model_comparison.ipynb` — side-by-side comparison (optional Gemini) and model export.